# Barrier-Gate Voltage Sweep

Compare the lateral electrostatic potential of an empty quantum dot with an accumulated (2DHG) quantum dot across the same barrier-gate voltages. Simulation execution is optional and disabled by default; the analysis can reuse existing completed sweep runs.

In [ ]:
import os
from pathlib import Path

from IPython.display import Markdown, display

from nextnanopp_tools import (
    discover_sweep_runs,
    load_sweep_outputs,
    plot_sweep_lines,
    run_sweep,
)

In [ ]:
START_PATH = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (START_PATH, *START_PATH.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not locate the repository root from {START_PATH}; "
        "expected a parent containing pyproject.toml and src/."
    )

RUN_SWEEP = False
SHOW_INTERACTIVE = False
SWEEP_VARIABLE = "V_BG"
BARRIER_VOLTAGES = [
    -2.0, -1.8, -1.6, -1.4, -1.2, -1.0, -0.8, -0.6,
    -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0,
    1.2, 1.4, 1.6, 1.8, 2.0, 5.0, 10.0, 25.0, 50.0,
]

# Set NEXTNANO_OUTPUT_ROOT to the desired nextnanopy outputdirectory.
# RUN_SWEEP passes this root explicitly and does not explicitly modify persisted configuration.
SIMULATION_OUTPUT_ROOT = Path(
    os.environ.get(
        "NEXTNANO_OUTPUT_ROOT",
        REPO_ROOT.parent / "qpu-local-outputs" / "refactoring" / "runs",
    )
).expanduser().resolve()
if SIMULATION_OUTPUT_ROOT == REPO_ROOT or REPO_ROOT in SIMULATION_OUTPUT_ROOT.parents:
    raise ValueError("SIMULATION_OUTPUT_ROOT must be outside the repository.")

EMPTY_QD_INPUT = REPO_ROOT / "configs/nextnano_inputs/baseline_single_qd/2d/Single_Empty_Quantum_Dot_2D.in"
ACCUMULATED_QD_INPUT = REPO_ROOT / "configs/nextnano_inputs/baseline_single_qd/2d/Single_Quantum_Dot_2D.in"
EMPTY_QD_SWEEP_ROOT = SIMULATION_OUTPUT_ROOT / f"{EMPTY_QD_INPUT.stem}_sweep__{SWEEP_VARIABLE}"
ACCUMULATED_QD_SWEEP_ROOT = SIMULATION_OUTPUT_ROOT / f"{ACCUMULATED_QD_INPUT.stem}_sweep__{SWEEP_VARIABLE}"

SELECTED_BIAS = "bias_00000"
SELECTED_OUTPUT = Path("potential_1d_x_QD.dat")
SELECTED_VARIABLE = None
REFERENCE_COORD = None
X_LIMITS = (-200.0, 200.0)
Y_LIMITS = None
STRICT_DISCOVERY = True

DELETE_OLD_FILES = True
DELETE_INPUT_FILES = False
OVERWRITE = True
SHOW_LOG = False
CONVERGENCE_CHECK = True
PARALLEL_LIMIT = 1

In [ ]:
CASES = {
    "empty": {
        "input_path": EMPTY_QD_INPUT,
        "sweep_root": EMPTY_QD_SWEEP_ROOT,
        "label": "Empty QD",
    },
    "accumulated": {
        "input_path": ACCUMULATED_QD_INPUT,
        "sweep_root": ACCUMULATED_QD_SWEEP_ROOT,
        "label": "Accumulated QD (2DHG)",
    },
}

In [ ]:
# run_sweep forwards outputdirectory to nextnanopy, which appends the native
# <input_stem>_sweep__<variable> directory represented in each CASES entry.
SWEEP_RESULTS = {}
if RUN_SWEEP:
    missing_inputs = [
        case["input_path"]
        for case in CASES.values()
        if not case["input_path"].is_file()
    ]
    if missing_inputs:
        missing_text = ", ".join(str(path) for path in missing_inputs)
        raise FileNotFoundError(f"Sweep input files were not found: {missing_text}")

    inconsistent_roots = [
        case["sweep_root"]
        for case in CASES.values()
        if case["sweep_root"].parent.expanduser().resolve()
        != SIMULATION_OUTPUT_ROOT
    ]
    if inconsistent_roots:
        raise RuntimeError(
            "Every case sweep root must share SIMULATION_OUTPUT_ROOT as its parent."
        )

    for case_name, case in CASES.items():
        sweep = run_sweep(
            case["input_path"],
            {SWEEP_VARIABLE: BARRIER_VOLTAGES},
            delete_old_files=DELETE_OLD_FILES,
            delete_input_files=DELETE_INPUT_FILES,
            overwrite=OVERWRITE,
            show_log=SHOW_LOG,
            convergenceCheck=CONVERGENCE_CHECK,
            parallel_limit=PARALLEL_LIMIT,
            execute_kwargs={"outputdirectory": str(case["sweep_root"].parent)},
        )
        actual_sweep_root = Path(sweep.sweep_output_directory).expanduser().resolve()
        expected_sweep_root = case["sweep_root"].expanduser().resolve()
        if actual_sweep_root != expected_sweep_root:
            raise RuntimeError(
                f"{case['label']} sweep was written to {actual_sweep_root}, "
                f"but discovery is configured for {expected_sweep_root}."
            )
        SWEEP_RESULTS[case_name] = sweep

In [ ]:
MANIFESTS = {
    case_name: discover_sweep_runs(
        case["sweep_root"],
        SWEEP_VARIABLE,
        bias=SELECTED_BIAS,
        required_outputs=(SELECTED_OUTPUT,),
        strict=STRICT_DISCOVERY,
    )
    for case_name, case in CASES.items()
}

In [ ]:
DISCOVERY_COLUMNS = [
    "sweep_value",
    "run_name",
    "complete",
    "bias_dir",
    "outputs_available",
    "error",
]

for case_name, manifest in MANIFESTS.items():
    display(
        Markdown(f"### {CASES[case_name]['label']} discovery"),
        manifest.loc[:, DISCOVERY_COLUMNS],
    )
    if manifest.empty:
        raise RuntimeError(f"No sweep runs were discovered for {case_name}.")
    unavailable = manifest.loc[
        ~manifest["complete"] | ~manifest["outputs_available"]
    ]
    if not unavailable.empty:
        failed_runs = ", ".join(unavailable["run_name"].tolist())
        raise RuntimeError(
            f"Incomplete runs or unavailable outputs for {case_name}: {failed_runs}"
        )

In [ ]:
LOADED_OUTPUTS = {
    case_name: load_sweep_outputs(
        MANIFESTS[case_name],
        SELECTED_OUTPUT,
        variable=SELECTED_VARIABLE,
    )
    for case_name in CASES
}

for case_name, outputs in LOADED_OUTPUTS.items():
    failures = outputs.loc[outputs["load_error"].notna()]
    if not failures.empty:
        display(
            Markdown(f"### {CASES[case_name]['label']} loading failures"),
            failures.loc[:, ["sweep_value", "run_name", "output_path", "load_error"]],
        )
        raise RuntimeError(f"Failed to load one or more outputs for {case_name}.")

In [ ]:
STATIC_FIGURES = {}
for case_name, case in CASES.items():
    figure = plot_sweep_lines(
        LOADED_OUTPUTS[case_name],
        variable=SELECTED_VARIABLE,
        interactive=False,
        title=f"{case['label']}: lateral potential versus {SWEEP_VARIABLE}",
        xlim=X_LIMITS,
        ylim=Y_LIMITS,
        reference_coord=REFERENCE_COORD,
    )
    STATIC_FIGURES[case_name] = figure
    display(figure)

In [ ]:
INTERACTIVE_FIGURES = {}
if SHOW_INTERACTIVE:
    for case_name, case in CASES.items():
        figure = plot_sweep_lines(
            LOADED_OUTPUTS[case_name],
            variable=SELECTED_VARIABLE,
            interactive=True,
            title=f"{case['label']}: lateral potential versus {SWEEP_VARIABLE}",
            xlim=X_LIMITS,
            ylim=Y_LIMITS,
            reference_coord=REFERENCE_COORD,
        )
        INTERACTIVE_FIGURES[case_name] = figure
        display(figure)

## Further 1D quantities

Change `SELECTED_OUTPUT` and `SELECTED_VARIABLE` to reuse the same workflow for another one-dimensional quantity, such as `potential_1d_x_QD.dat` or a 1D HH band-edge output along the selected axis.